# Audio Semantic Diagnostics — forward-graft display surface

**DISPLAY-ONLY.** This notebook reimplements NO tempo / beat / onset / saliency / AP DSP.
Every numeric value shown is produced upstream by the **unmodified firmware C++** replayed
through the canonical host harness, and dumped into the trajectory / A/B JSON artifacts under
`build/audio-semantic-metrics/`. This surface only *loads and plots* those artifacts via
`diag_helpers.py`. The host harness is the source of truth; acceptance numbers come from the
JSON artifacts, never from notebook logic.

Mandatory annotations: every host-modelled front-end plot carries
`MODELLED FRONT-END / HOST CEILING — not guaranteed to match device AP_STREAM.`; every
host/device overlay labels the host trace, the device AP_STREAM trace, and whether the
capture matches the track.

## Controls (parameters)

Only **display** controls live here. **Forbidden tuning controls are deliberately ABSENT** —
there is NO confidence threshold knob, NO PLL Kp/Ki, NO onset threshold, NO refractory, NO
prior width, NO scoring weights. This is a diagnostic surface, not a tuning playground; tuning
happens in the host harness / firmware, never here. `lock_threshold` below is only a *reference
line* drawn at the firmware's known SB_LOCK_CONFIDENCE (0.60) for reading the plot — it changes
no computation.

In [1]:
# --- papermill parameters cell (display controls ONLY) ---
TRACK_ID = "7vevOMWY6MY"          # which exported track to inspect
RUN_MODE = "final"                # 'baseline' | 'final' | 'ab'
TIME_WINDOW_S = None             # (lo, hi) seconds, or None for full clip
BEAT_TOL_MS = 70                 # ±tolerance band for the beat-vs-GT view (display)
SHOW_GT = True
SHOW_BEAT_TICK = True
SHOW_APSTREAM = False            # overlay legacy [AP] device stream if available
APSTREAM_PATH = None              # path to a captured log with [AP] stream
SHOW_APCAPTURE = False           # plot APCAP summary windows if available
APCAPTURE_PATH = None             # path to a captured log with [APCAP]
SHOW_AGC_STREAM = False          # plot AGC debug stream if available
AGC_STREAM_PATH = None            # path to a captured log with sbs((agc_debug=...))
CONF_RAW_VS_SMOOTH = True        # show smoothed conf alongside raw (display smoother)
BEAT_RAW_VS_DEDUP = "dedup"      # 'dedup' | 'raw' beat_tick view
NORMALIZE_NOVELTY = True         # normalise novelty to [0,1] for plotting


In [2]:
import sys, os
from pathlib import Path
# locate repo root: env override (set by render_via_nbconvert.py) > cwd search > scan upward.
def _find_root():
    env = os.environ.get('DIAG_REPO_ROOT')
    if env and (Path(env) / 'notebooks' / 'diag_helpers.py').exists():
        return Path(env)
    here = Path.cwd()
    for cand in [here, here.parent, *here.parents]:
        if (cand / 'notebooks' / 'diag_helpers.py').exists():
            return cand
    return here
ROOT = _find_root()
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'scripts' / 'regression-harness'))
import matplotlib.pyplot as plt
import diag_helpers as dh

# resolve the trajectory for the selected track/run. In 'ab' mode default the
# single-run views to 'final' (the graft) and compare against baseline in section 11.
_run = 'final' if RUN_MODE == 'ab' else RUN_MODE
traj = dh.load_trajectory(TRACK_ID, _run)
traj_baseline = dh.load_trajectory(TRACK_ID, 'baseline')
ab_baseline = dh.load_ab('baseline')
ab_final = dh.load_ab('final')
ab_onset = dh.load_ab('onset_v2_ab')
ab_chord = dh.load_ab('chord_v2_ab')

_capture_path = APSTREAM_PATH or APCAPTURE_PATH or AGC_STREAM_PATH
_capture = dh.load_apstream(_capture_path, track_id=TRACK_ID) if (
    _capture_path is not None and (SHOW_APSTREAM or SHOW_APCAPTURE or SHOW_AGC_STREAM)
) else None
apstream = _capture.get('ap_stream') if isinstance(_capture, dict) else None
ap_capture = _capture.get('ap_capture') if isinstance(_capture, dict) else None
agc_debug = _capture.get('agc_debug') if isinstance(_capture, dict) else None

WIN = tuple(TIME_WINDOW_S) if TIME_WINDOW_S else None
print(f'TRACK_ID={TRACK_ID}  RUN_MODE={RUN_MODE}  run-views={_run}')
print('trajectory loaded:', traj is not None, '| baseline:', traj_baseline is not None)
print('available exported tracks:', dh.list_trajectory_tracks())
if _capture is not None:
    print('capture loaded:', _capture_path,
          f'AP_STREAM parsed={apstream.get("n_parsed") if isinstance(apstream, dict) else 0},',
          f'APCAP parsed={(ap_capture or {}).get("n_parsed", 0)},',
          f'AGC parsed={(agc_debug or {}).get("n_parsed", 0)}')
elif SHOW_APSTREAM or SHOW_APCAPTURE or SHOW_AGC_STREAM:
    print('device telemetry requested but not loaded (missing paths or parseable capture).')


TRACK_ID=7vevOMWY6MY  RUN_MODE=final  run-views=final
trajectory loaded: True | baseline: True
available exported tracks: ['7vevOMWY6MY', 'Qa1AqKxakRM', 'T-sxSd1uwoU', 'WxnN05vOuSM', 'c7tOAGY59uQ', 'r9DBFTZTKPI']


## 1. Provenance
Branch / commit / flags / corpus / annotation source / harness command / sample rate / hop / update rate — all from the trajectory artifact's `provenance` block (not asserted here).

In [3]:
if traj is None:
    print(f'NO TRAJECTORY for {TRACK_ID} [{_run}] — run export_diagnostic_trajectories.py first.')
else:
    p = traj['provenance']
    print(f"branch          : {p['branch']}")
    print(f"head            : {p['head']}")
    print(f"run / flags     : {traj['run']}  {p['flags']}")
    print(f"sample_rate_hz  : {p['sample_rate_hz']}   hop: {p['hop']}")
    print(f"ap_frame_hz     : {p['ap_frame_hz']:.3f}   novelty_rate_hz (/3): {p['novelty_rate_hz']:.3f}")
    print(f"corpus_path     : {p['corpus_path']}")
    print(f"gt_source       : {p['gt_source']}")
    print(f"harness_cmd     : {p['harness_cmd']}")
    print(f"curated_reason  : {p['curated_reason']}")
    print(f"GT bpm / title  : {traj['gt']['bpm']}  {traj['gt']['title']}  ({traj['gt']['genre']})")

branch          : wip/audio-saliency-recovery
head            : fb83bd1f3301e721318fec76265b0468482372ad
run / flags     : final  ['SB_TEMPO_CONF_V2', 'SB_TEMPO_FLYWHEEL_V2']
sample_rate_hz  : 12800   hop: 96
ap_frame_hz     : 133.333   novelty_rate_hz (/3): 44.444
corpus_path     : /Users/spectrasynq/SensoryBridge-main 9/Lightwave-Ledstrip/firmware-v3/test/music_corpus/harmonixset/esv11_benchmark/audio_12k8/7vevOMWY6MY_12k8.wav
gt_source       : /Users/spectrasynq/Workspace_Management/Software/K1.reinvented/Implementation.plans/harmonixset-main/dataset
harness_cmd     : novelty_from_wav.wav_to_novelty -> tempo_replay.build_binary(defines=['SB_TEMPO_CONF_V2', 'SB_TEMPO_FLYWHEEL_V2']).replay_stdin  ||  onset: onset_v2_replay.build_replay(['SB_ONSET_V2']).run_replay
curated_reason  : clean mid-tempo Pop 128 BPM 4/4 (0877 Whatchamacallit) — clean-control
GT bpm / title  : 128.0  Whatchamacallit  (Pop)


## 2. Track metadata + waveform
The corpus audio samples themselves (source music, not a modelled signal).

In [4]:
fig, ax = plt.subplots(figsize=(12, 2.6))
dh.plot_waveform(TRACK_ID, window_s=WIN, ax=ax)
plt.show()

## 3. Modelled front-end novelty
Spectral-flux onset curve fed to sb_tempo (`novelty_from_wav`). HOST CEILING.

In [5]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 2.8))
    dh.plot_novelty(traj, window_s=WIN, normalize=NORMALIZE_NOVELTY, ax=ax)
    plt.show()

## 4. AP_STREAM overlay
Device capture vs host-modelled bpm. If `SHOW_APSTREAM` is off or the capture is absent, a clear *no device capture — host-modelled only* note is shown. ~1 Hz AP_STREAM under-samples per-beat phase (fine for bpm/conf/lock trend).

In [6]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 3.0))
    dh.plot_apstream_overlay(traj, apstream, show=SHOW_APSTREAM, ax=ax)
    plt.show()
    if apstream is not None:
        print('AP_STREAM parsed:', apstream['n_parsed'], 'skipped:', apstream['n_skipped'])
        for w in apstream['warnings']:
            print('  WARN:', w)

## 4a. APCAP capture summary
AP capture window summary from `[APCAP]` (frames + max_raw + peak_scaled + follower + spec/chroma markers).
This is host-independent front-end capture telemetry, not AGC-band debug.


In [ ]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 2.8))
    dh.plot_ap_capture_summary(ap_capture, window_s=WIN, ax=ax)
    plt.show()
    if ap_capture is not None:
        print('APCAP parsed:', ap_capture.get('n_parsed', 0), 'skipped:', ap_capture.get('n_skipped', 0))
        last = ap_capture.get('last') or {}
        print('  APCAP last:', last.get('frames', 'n/a'), 'spec_argmax', last.get('spec_argmax', 'n/a'),
              'follower', last.get('follower_mean', 'n/a'))
elif not SHOW_APCAPTURE:
    print('SHOW_APCAPTURE is off.')


## 4b. AGC debug stream
Per-band AGC medians over time from `sbs((agc_debug=...))` (`energy`, `gain`, `threshold`, `floor`).


In [ ]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 2.8))
    dh.plot_agc_debug(agc_debug, window_s=WIN, ax=ax)
    plt.show()
    if agc_debug is not None:
        print('AGC parsed:', agc_debug.get('n_parsed', 0), 'skipped:', agc_debug.get('n_skipped', 0))
        s = agc_debug.get('samples', {})
        for k in ('energy', 'gain', 'threshold', 'floor'):
            vals = s.get(k, [])
            print('  ', k, 'last:', vals[-1] if vals else 'n/a', '| mean:', (sum(vals) / len(vals)) if vals else 'n/a')
elif not SHOW_AGC_STREAM:
    print('SHOW_AGC_STREAM is off.')


## 5. Tempogram / ACF / comb heatmap
sb_tempo's raw Goertzel-over-novelty periodicity surface (the `RAWSPEC` artifact, bin *i* = 60+*i* BPM). If the artifact is absent, the host BPM trajectory is shown instead.

In [7]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 3.0))
    dh.plot_tempogram(traj, ax=ax)
    plt.show()

## 6. BPM over time vs GT
Detected BPM trajectory against the gold BPM line (and ½×/2× octave guides).

In [8]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 3.0))
    dh.plot_bpm_vs_gt(traj, window_s=WIN, show_gt=SHOW_GT, ax=ax)
    plt.show()

## 7. Confidence + lock overlay
Raw confidence (+ display smoother), the lock-threshold reference line, and lock/unlock edge events (green=lock, red=unlock).

In [9]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 3.0))
    dh.plot_conf_lock(traj, window_s=WIN, lock_threshold=0.60,
                      raw_vs_smooth=CONF_RAW_VS_SMOOTH, ax=ax)
    plt.show()

## 8. Predicted beat_tick vs GT
Tolerance band (±`BEAT_TOL_MS`), false-positive (red) + missed-GT (red ✗) highlighting (display-derived), and a density note including the raw-vs-dedup tick view.

In [10]:
if traj and SHOW_BEAT_TICK:
    fig, ax = plt.subplots(figsize=(12, 2.6))
    dh.plot_beat_vs_gt(traj, window_s=WIN, beat_tol_ms=BEAT_TOL_MS, show_gt=SHOW_GT,
                       raw_vs_dedup=BEAT_RAW_VS_DEDUP, ax=ax)
    plt.show()
    # density note: raw vs dedup tick counts (both are harness views, not metrics)
    raw_n = int(dh.beat_tick_times_ms(traj, dedup=False).size)
    dd_n = int(dh.beat_tick_times_ms(traj, dedup=True).size)
    print(f'beat_tick density — raw: {raw_n} ticks, dedup: {dd_n} ticks '
          f'(raw carries the 133 Hz stale-republish; dedup is the leading-edge view)')
elif traj:
    print('SHOW_BEAT_TICK is off.')

beat_tick density — raw: 25 ticks, dedup: 25 ticks (raw carries the 133 Hz stale-republish; dedup is the leading-edge view)


## 9. Onset channels
transient / kick / snare / hihat events (final run only; V2-only). Beat/downbeat are sb_tempo-owned (section 8); **downbeat is not available** from any harness.

In [11]:
if traj:
    fig, ax = plt.subplots(figsize=(12, 2.6))
    dh.plot_onset_channels(traj, window_s=WIN, ax=ax)
    plt.show()

## 10. Harmonic / chord state
Chord-detection confidences + harmonic-saliency-axis A/B from `chord_v2_ab.json`. NOTE: this is a **synthetic labelled-fixture** A/B, not a per-track real-music chord stream (no per-track chord events exist; see trajectory `chord_events_note`).

In [12]:
fig, ax = plt.subplots(figsize=(10, 4.0))
dh.plot_harmonic_state(ab_chord, ax=ax)
plt.show()
if ab_chord:
    hsa = ab_chord.get('harmonic_saliency_axis', {})
    print('harmonic axis transitions (synthetic):',
          'root-change spike', hsa.get('v2_raw_root_change_spike'),
          '| type-change spike', hsa.get('v2_raw_type_change_spike'),
          '| incumbent dead', hsa.get('incumbent_peak_dead'))

harmonic axis transitions (synthetic): root-change spike 1.0 | type-change spike 0.6 | incumbent dead 0.0


## 11. A/B comparison (baseline vs final)
From `baseline.json` / `final.json` / `onset_v2_ab.json` aggregates: confidence reachability, false-lock rate, beat-F / continuity proxy, onset proxy, aggregate deltas. **All numbers come from the JSON artifacts** — the notebook applies no acceptance logic.

In [13]:
fig, ax = plt.subplots(figsize=(11, 4.5))
dh.plot_ab_summary(ab_baseline, ab_final, ab_onset, ax=ax)
plt.show()

print('=== headline aggregate deltas (baseline -> final) ===')
for label, b, f, d in dh.ab_aggregate_rows(ab_baseline, ab_final):
    arrow = '+' if d >= 0 else ''
    print(f'  {label:48} {b:8.4f} -> {f:8.4f}   ({arrow}{d:.4f})')

print('\n=== false-lock (silence/noise) lock_frac, max_conf ===')
for name, probe, lf, mc in dh.false_lock_rows(ab_baseline, ab_final):
    print(f'  {name:8} {probe:8} lock_frac={lf}  max_conf={mc}')

op = dh.onset_proxy_rows(ab_onset)
if op:
    print('\n=== onset proxy (P/R/F1) incumbent vs v2 ===')
    print('  incumbent:', op['incumbent_pr'])
    print('  v2       :', op['v2_pr'])
    print('  AGC-clamp survival:', op['agc'])

=== headline aggregate deltas (baseline -> final) ===
  conf reachability (frac tracks reach .60)          1.0000 ->   1.0000   (+0.0000)
  frac tracks ever locked                            1.0000 ->   1.0000   (+0.0000)
  median settled conf                                0.0407 ->   0.6204   (+0.5797)
  median max conf                                    1.0000 ->   0.9621   (-0.0379)
  beat-F (metrical-best, mean)                       0.0385 ->   0.0314   (-0.0070)
  beat-F (x1, mean)                                  0.0307 ->   0.0261   (-0.0045)
  CMLt-like continuity (mean)                        0.0099 ->   0.0081   (-0.0018)
  AMLt-like continuity (mean)                        0.0101 ->   0.0085   (-0.0016)
  frac tracks density-in-band                        0.0556 ->   0.9722   (+0.9167)
  median pred density (Hz)                           4.6805 ->   1.4116   (-3.2689)

=== false-lock (silence/noise) lock_frac, max_conf ===
  baseline silence  lock_frac=0.0  max_conf=0.0
  

## 12. Interpretation
What improved, what is host-or-device-uncertain, and what needs the final device check.
All claims below are read off the artifacts shown above — this cell asserts nothing new.

In [14]:
rows = dh.ab_aggregate_rows(ab_baseline, ab_final)
improved = [r for r in rows if r[3] > 0.001]
regressed = [r for r in rows if r[3] < -0.001]
print('WHAT IMPROVED (final vs baseline, from JSON aggregates):')
for label, b, f, d in improved:
    print(f'  + {label}: {b:.4f} -> {f:.4f}  (+{d:.4f})')
print('\nWHAT REGRESSED / UNCHANGED-OR-DOWN:')
for label, b, f, d in regressed:
    print(f'  - {label}: {b:.4f} -> {f:.4f}  ({d:.4f})')
print('\nHOST-OR-DEVICE-UNCERTAIN:')
print('  * Every front-end / tempo / beat plot above is the MODELLED HOST CEILING')
print('    (novelty_from_wav onset curve, not the device AGC-clamped GDFT novelty).')
print('  * No device AP_STREAM capture is overlaid unless SHOW_APSTREAM + APSTREAM_PATH')
print('    point at a real capture matched to this track.')
print('  * Harmonic/chord state is a SYNTHETIC labelled-fixture A/B, not real-music chords.')
print('\nNEEDS THE FINAL DEVICE CHECK:')
print('  * On-device AP_STREAM bpm/conf/lock vs this host ceiling on the SAME track.')
print('  * Per-beat phase alignment (AP_STREAM is ~1 Hz — too coarse; needs device beat log).')
print('  * Onset per-band behaviour under the real device broadband AGC clamp.')

WHAT IMPROVED (final vs baseline, from JSON aggregates):
  + median settled conf: 0.0407 -> 0.6204  (+0.5797)
  + frac tracks density-in-band: 0.0556 -> 0.9722  (+0.9167)

WHAT REGRESSED / UNCHANGED-OR-DOWN:
  - median max conf: 1.0000 -> 0.9621  (-0.0379)
  - beat-F (metrical-best, mean): 0.0385 -> 0.0314  (-0.0070)
  - beat-F (x1, mean): 0.0307 -> 0.0261  (-0.0045)
  - CMLt-like continuity (mean): 0.0099 -> 0.0081  (-0.0018)
  - AMLt-like continuity (mean): 0.0101 -> 0.0085  (-0.0016)
  - median pred density (Hz): 4.6805 -> 1.4116  (-3.2689)

HOST-OR-DEVICE-UNCERTAIN:
  * Every front-end / tempo / beat plot above is the MODELLED HOST CEILING
    (novelty_from_wav onset curve, not the device AGC-clamped GDFT novelty).
  * No device AP_STREAM capture is overlaid unless SHOW_APSTREAM + APSTREAM_PATH
    point at a real capture matched to this track.
  * Harmonic/chord state is a SYNTHETIC labelled-fixture A/B, not real-music chords.

NEEDS THE FINAL DEVICE CHECK:
  * On-device AP_STREAM